In [ ]:
!pip install -q -U transformers accelerate requests

import os
import json
import glob
import time
import torch
from collections import Counter
from PIL import Image
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-VL-7B-Instruct"

processor = AutoProcessor.from_pretrained(
    MODEL_NAME,
    min_pixels=256 * 28 * 28,
    max_pixels=1024 * 28 * 28,
)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    max_memory={0: "12GiB", 1: "12GiB"},
)
model.eval()
for p in model.parameters():
    p.requires_grad_(False)

ln_f = model.model.language_model.norm
lm_head = model.lm_head
layers = model.model.language_model.layers

n_layer = model.config.text_config.num_hidden_layers
n_head = model.config.text_config.num_attention_heads
hidden_dim = model.config.text_config.hidden_size
head_dim = layers[0].self_attn.o_proj.in_features // n_head
rms_eps = model.config.text_config.rms_norm_eps
assert len(layers) == n_layer
print(n_layer, "language model layers,", n_head, "heads,", head_dim, "head dim")

In [ ]:
AMBER_IMAGE_DIR = "/kaggle/input/datasets/nocturnalnerd18/amber-hallucination/image"
VG_IMAGE_DIRS = ["/kaggle/temp", "/kaggle/temp/VG_100K_2"]


def load_image(path):
    return Image.open(path).convert("RGB")


def build_inputs(image, text):
    conversation = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": text}]}]
    prompt_text = processor.apply_chat_template(conversation, add_generation_prompt=True)
    return processor(text=prompt_text, images=image, return_tensors="pt").to(model.device, torch.float16)


def find_vg_image_path(image_id: str) -> str:
    for d in VG_IMAGE_DIRS:
        candidate = os.path.join(d, f"{image_id}.jpg")
        if os.path.exists(candidate):
            return candidate
    return None


def resolve_image_path(q: dict) -> str:
    qid = q["question_id"]
    if qid.startswith("amber_"):
        return os.path.join(AMBER_IMAGE_DIR, f"{q['image_id']}.jpg")
    elif qid.startswith("reefknot_"):
        return find_vg_image_path(q["image_id"])
    return None

In [ ]:
INPUT = "/kaggle/input/datasets/nocturnalnerd18"
USERNAME = "nocturnalnerd18"
TYPES = ["attribute", "relation"]
PAIRS = {"amber_": (9454, 2753), "reefknot_": (9693, 2152)}
CAND_IDS = [PAIRS["amber_"][0], PAIRS["amber_"][1], PAIRS["reefknot_"][0], PAIRS["reefknot_"][1]]

with open(f"{INPUT}/vlm-questions/all_questions.json") as f:
    all_questions = json.load(f)

assert len(all_questions) == 17492, "attach vlm-questions version 2"
print(len(all_questions), "questions loaded")
for prefix, (yes_id, no_id) in PAIRS.items():
    print(prefix, processor.tokenizer.convert_ids_to_tokens([yes_id, no_id]))
assert processor.tokenizer.convert_ids_to_tokens(CAND_IDS) == ["Yes", "No", "yes", "no"]

In [ ]:
_state = {"prefix": None}
_u_cache = {}

U = {
    prefix: ln_f.weight.float().cpu() * (lm_head.weight[yes_id] - lm_head.weight[no_id]).float().cpu()
    for prefix, (yes_id, no_id) in PAIRS.items()
}


def u_on(device):
    key = (_state["prefix"], device)
    if key not in _u_cache:
        _u_cache[key] = U[_state["prefix"]].to(device)
    return _u_cache[key]


HEAD_PROJ = {}
with torch.no_grad():
    for prefix in PAIRS:
        _state["prefix"] = prefix
        HEAD_PROJ[prefix] = []
        for layer in layers:
            w = layer.self_attn.o_proj.weight
            HEAD_PROJ[prefix].append(
                w.float().T.contiguous().view(n_head, head_dim, hidden_dim) @ u_on(w.device)
            )

_dla_cache = {}


def _embed_hook(module, args, kwargs):
    h = args[0] if args else kwargs["hidden_states"]
    _dla_cache["embed"] = h[0, -1].float() @ u_on(h.device)


def _make_attn_hook(l):
    def hook(module, args):
        z = args[0][0, -1].float().reshape(n_head, head_dim)
        _dla_cache[f"attn_{l}"] = (z * HEAD_PROJ[_state["prefix"]][l]).sum(-1)
    return hook


def _make_mlp_hook(l):
    def hook(module, args, output):
        _dla_cache[f"mlp_{l}"] = output[0, -1].float() @ u_on(output.device)
    return hook


def _final_norm_hook(module, args):
    h = args[0][0, -1].float()
    _dla_cache["resid_dot"] = h @ u_on(h.device)
    _dla_cache["rms"] = torch.sqrt(h.pow(2).mean() + rms_eps)


_hook_handles = [layers[0].register_forward_pre_hook(_embed_hook, with_kwargs=True)]
for l, layer in enumerate(layers):
    _hook_handles.append(layer.self_attn.o_proj.register_forward_pre_hook(_make_attn_hook(l)))
    _hook_handles.append(layer.mlp.register_forward_hook(_make_mlp_hook(l)))
_hook_handles.append(ln_f.register_forward_pre_hook(_final_norm_hook))
print(len(_hook_handles), "hooks registered")

In [ ]:
@torch.no_grad()
def dla_question(image_path, question_text, prefix):
    _state["prefix"] = prefix
    yes_id, no_id = PAIRS[prefix]
    image = load_image(image_path)
    inputs = build_inputs(image, text=question_text)

    _dla_cache.clear()
    outputs = model(**inputs)
    last_logits = outputs.logits[0, -1].float().cpu()
    top_id = int(last_logits.argmax())

    return {
        "embed": _dla_cache["embed"].cpu(),
        "attn": torch.stack([_dla_cache[f"attn_{l}"].cpu() for l in range(n_layer)]),
        "mlp": torch.stack([_dla_cache[f"mlp_{l}"].cpu() for l in range(n_layer)]),
        "resid_dot": _dla_cache["resid_dot"].cpu(),
        "rms": _dla_cache["rms"].cpu(),
        "logit_diff": last_logits[yes_id] - last_logits[no_id],
        "cand_logits": last_logits[CAND_IDS].clone(),
        "top_id": top_id,
        "in_pair": top_id in (yes_id, no_id),
        "answer_text": processor.tokenizer.decode(top_id).strip(),
    }

In [ ]:
DLA_DIR = "/kaggle/working/dla_qwen"
os.makedirs(DLA_DIR, exist_ok=True)

BATCH_SAVE_SIZE = 200
worst = {"rel_err_resid": 0.0, "err_logit": 0.0, "nonfinite": 0}


def collect(halluc_type):
    questions_subset = [q for q in all_questions if q["hallucination_type"] == halluc_type and q["answer_format"] == "yes_no"]
    paths = sorted(glob.glob(os.path.join(DLA_DIR, f"{halluc_type}_batch_*.pt")))
    done_ids = set()
    for path in paths:
        done_ids.update(e["question_id"] for e in torch.load(path))
    todo = [q for q in questions_subset if q["question_id"] not in done_ids]
    print(f"{halluc_type}: {len(questions_subset)} questions, {len(done_ids)} already done, {len(todo)} to run")

    batch_entries, batch_index, skipped = [], len(paths), 0
    t_start = time.time()

    for i, q in enumerate(todo):
        image_path = resolve_image_path(q)
        prefix = q["question_id"].split("_")[0] + "_"
        if image_path is None or not os.path.exists(image_path) or prefix not in PAIRS:
            skipped += 1
            continue

        entry = dla_question(image_path, q["question_text"], prefix)
        total = entry["embed"] + entry["attn"].sum() + entry["mlp"].sum()
        scale = entry["embed"].abs() + entry["attn"].abs().sum() + entry["mlp"].abs().sum()
        if torch.isfinite(total) and torch.isfinite(entry["logit_diff"]):
            worst["rel_err_resid"] = max(worst["rel_err_resid"], ((total - entry["resid_dot"]).abs() / scale).item())
            worst["err_logit"] = max(worst["err_logit"], (total / entry["rms"] - entry["logit_diff"]).abs().item())
        else:
            worst["nonfinite"] += 1
        entry.update({k: q[k] for k in ("image_id", "question_id", "ground_truth_answer", "answer_format", "hallucination_type")})
        entry["prefix"] = prefix
        batch_entries.append(entry)

        if len(batch_entries) >= BATCH_SAVE_SIZE:
            torch.save(batch_entries, os.path.join(DLA_DIR, f"{halluc_type}_batch_{batch_index}.pt"))
            batch_entries, batch_index = [], batch_index + 1

        if i % 100 == 0 and i > 0:
            elapsed = time.time() - t_start
            avg = elapsed / (i + 1)
            remaining = avg * (len(todo) - i - 1)
            print(f"[{i+1}/{len(todo)}] {q['question_id']}: {entry['answer_text']!r} "
                  f"({avg:.2f}s/question avg, ETA {remaining/60:.1f} min, skipped: {skipped}, nonfinite: {worst['nonfinite']})")

    if batch_entries:
        torch.save(batch_entries, os.path.join(DLA_DIR, f"{halluc_type}_batch_{batch_index}.pt"))

    print(f"done -- {halluc_type}, {skipped} skipped, worst rel err resid {worst['rel_err_resid']:.5f}, "
          f"worst err logit {worst['err_logit']:.4f}, nonfinite {worst['nonfinite']}, {(time.time()-t_start)/60:.1f} min")
    return skipped

In [ ]:
skipped = collect("attribute")
assert skipped == 0, f"{skipped} questions skipped"

In [ ]:
!wget https://cs.stanford.edu/people/rak248/VG_100K/images.zip -O /kaggle/working/VG_100K.zip
!wget https://cs.stanford.edu/people/rak248/VG_100K_2/images2.zip -O /kaggle/working/VG_100K_2.zip
!unzip -q -o /kaggle/working/VG_100K.zip -d /kaggle/temp/
!unzip -q -o /kaggle/working/VG_100K_2.zip -d /kaggle/temp/
for zpath in ["/kaggle/working/VG_100K.zip", "/kaggle/working/VG_100K_2.zip"]:
    if os.path.exists(zpath):
        os.remove(zpath)
print("Visual Genome downloaded, extracted, zips removed")

In [ ]:
skipped = collect("relation")
assert skipped == 0, f"{skipped} questions skipped"

In [ ]:
for t in TYPES:
    entries = []
    for path in glob.glob(os.path.join(DLA_DIR, f"{t}_batch_*.pt")):
        entries.extend(torch.load(path))
    ids = [e["question_id"] for e in entries]
    expected = {q["question_id"] for q in all_questions if q["hallucination_type"] == t and q["answer_format"] == "yes_no"}
    print(f"{t:10s} entries {len(ids)} (expected {len(expected)}) | missing {len(expected - set(ids))} | duplicates {len(ids) - len(set(ids))}")
    assert set(ids) == expected and len(ids) == len(expected), f"{t} incomplete"
    for prefix in PAIRS:
        sub = [e for e in entries if e["prefix"] == prefix]
        if sub:
            print(f"  {prefix:10s} {len(sub)} entries | top-1 in pair {sum(e['in_pair'] for e in sub) / len(sub):.4f} | "
                  f"yes {sum(e['logit_diff'] > 0 for e in sub) / len(sub):.4f}")

print(f"worst relative err vs final residual: {worst['rel_err_resid']:.5f}")
print(f"worst abs err vs model logit diff: {worst['err_logit']:.4f}")
print(f"non-finite entries: {worst['nonfinite']}")
assert worst["nonfinite"] == 0, "non-finite values"
assert worst["rel_err_resid"] < 0.05, "components do not sum to the final residual"
assert worst["err_logit"] < 0.5, "components do not reproduce the logit difference"
print("collection verified")

In [ ]:
with open(os.path.join(DLA_DIR, "dla_meta.json"), "w") as f:
    json.dump({
        "model": MODEL_NAME, "pairs": PAIRS, "cand_ids": CAND_IDS, "n_layer": n_layer, "n_head": n_head,
        "worst_rel_err_resid": worst["rel_err_resid"], "worst_err_logit": worst["err_logit"],
    }, f)

with open(os.path.join(DLA_DIR, "dataset-metadata.json"), "w") as f:
    json.dump({
        "title": "dla-qwen",
        "id": f"{USERNAME}/dla-qwen",
        "subtitle": "Direct logit attribution components for Qwen2.5-VL yes/no questions",
        "description": "Per-question direct logit attribution for Qwen2.5-VL-7B on every attribute and relation yes/no question in vlm-questions version 2. "
                       "embed, attn [layer, head], mlp [layer] and resid_dot are raw projections onto the Yes minus No direction "
                       "with the final norm gain folded in. Divide by rms to get logit units. The direction uses the capitalised pair for amber_ "
                       "questions and the lowercase pair for reefknot_ questions. logit_diff is the model's own gap for that pair, "
                       "cand_logits holds the last-position logits of [Yes, No, yes, no], and in_pair marks rows whose top-1 token is the pair token.",
        "licenses": [{"name": "CC0-1.0"}],
    }, f)

!kaggle datasets create -p {DLA_DIR}